# Run the T-ChIC modality-prediction benchmark

This notebook runs the OpenProblems `task_predict_modality` benchmark on the
prepared T-ChIC development dataset.

The first benchmark task is:

**H3K4me3 → normalized unspliced RNA**

The initial control method is `mean_per_gene`.

The benchmark inputs are generated by `02_prepare_tchic_benchmark.ipynb`.

For this development run:

- the 70% training partition is used for training;
- the 15% validation partition occupies the benchmark `test_*` slots;
- the final 15% test partition remains untouched.

Scores produced here are therefore **development/validation scores**, not final
test scores.

The generated OpenProblems workflow itself is left unmodified. A small runtime
configuration is generated below only to select working container images and
provide the Python environment required by the correlation metric.


## 1. Imports and repository paths

In [1]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import pandas as pd
import yaml

cwd = Path.cwd().resolve()

if cwd.name == "notebooks" and (cwd.parent / "src").is_dir():
    repo_root = cwd.parent
elif (cwd / "src").is_dir() and (cwd / "notebooks").is_dir():
    repo_root = cwd
else:
    raise RuntimeError(
        f"Could not determine repository root from working directory: {cwd}"
    )

prepared_dir = (
    repo_root
    / "outputs"
    / "tchic"
    / "prepared"
    / "h3k4me3_genelevel_dev"
)

workflow_dir = (
    repo_root
    / "target_benchmark_complete"
    / "nextflow"
    / "workflows"
    / "run_benchmark"
)

runtime_dir = repo_root / "outputs" / "tchic" / "runtime"

run_root = (
    repo_root
    / "outputs"
    / "tchic"
    / "runs"
    / "h3k4me3_genelevel_dev"
    / "mean_per_gene"
)

work_dir = run_root / "work"
publish_dir = run_root / "publish"
trace_path = run_root / "trace.txt"

runtime_config = runtime_dir / "mean_per_gene.config"
command_path = run_root / "run_command.sh"

for filename in [
    "train_mod1.h5ad",
    "train_mod2.h5ad",
    "test_mod1.h5ad",
    "test_mod2.h5ad",
]:
    assert (prepared_dir / filename).exists()

assert workflow_dir.exists()

runtime_dir.mkdir(parents=True, exist_ok=True)
run_root.mkdir(parents=True, exist_ok=True)

print("Repository:", repo_root)
print("Prepared data:", prepared_dir)
print("Workflow:", workflow_dir)
print("Run directory:", run_root)


Repository: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation
Prepared data: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/prepared/h3k4me3_genelevel_dev
Workflow: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/target_benchmark_complete/nextflow/workflows/run_benchmark
Run directory: /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/runs/h3k4me3_genelevel_dev/mean_per_gene


## 2. Record software and repository provenance

The Git commit and relevant software versions are recorded before running the
benchmark. A non-empty Git status is also shown so that uncommitted work is
visible rather than hidden.

In [2]:
def run_capture(command):
    result = subprocess.run(
        command,
        cwd=repo_root,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=True,
    )
    return result.stdout.strip()


git_commit = run_capture(
    ["git", "rev-parse", "HEAD"]
)

git_status = run_capture(
    ["git", "status", "--short"]
)

nextflow_version = run_capture(
    ["nextflow", "-version"]
)

print("Git commit:")
print(git_commit)

print()
print("Git status:")
print(git_status if git_status else "(clean)")

print()
print("Python:")
print(sys.executable)
print(sys.version)

print()
print("Nextflow:")
print(nextflow_version)


Git commit:
6a6b063e9de62dc1430f96aa0a9d69e8e5a06117

Git status:
M notebooks/01_data_split_design.ipynb
?? notebooks/02_prepare_tchic_benchmark.ipynb
?? notebooks/03_run_tchic_benchmark.ipynb

Python:
/home/michiel/miniforge3/envs/thesis/bin/python
3.10.20 | packaged by conda-forge | (main, Mar  5 2026, 16:42:22) [GCC 14.3.0]

Nextflow:
N E X T F L O W
      version 26.04.4 build 12445
      created 17-06-2026 16:30 UTC (18:30 CEST)
      cite doi:10.1038/nbt.3820
      http://nextflow.io


## 3. Generate the runtime configuration

The generated benchmark currently refers to development container tags that
are not available for the required components.

The working runtime configuration therefore explicitly selects:

- `openproblems/base_python:1` for `mean_per_gene`;
- the `build_main` MSE metric image;
- the `build_main` correlation metric image.

The correlation component also uses R/reticulate. The active Python executable
and Conda installation are exposed inside the container dynamically instead of
hard-coding a user-specific path.

These settings affect the execution environment, not the method or metric
algorithms.


In [3]:
python_executable = Path(sys.executable).resolve()

conda_prefix_string = os.environ.get("CONDA_PREFIX")

if conda_prefix_string is None:
    raise RuntimeError(
        "CONDA_PREFIX is not defined. Run this notebook from the thesis Conda environment."
    )

conda_prefix = Path(conda_prefix_string).resolve()

if conda_prefix.parent.name == "envs":
    conda_root = conda_prefix.parent.parent
else:
    conda_root = conda_prefix

config_text = f"""process {{
  withName: '.*mean_per_gene_process' {{
    container = 'openproblems/base_python:1'
  }}

  withName: '.*mse_process' {{
    container = 'ghcr.io/openproblems-bio/task_predict_modality/metrics/mse:build_main'
  }}

  withName: '.*correlation_process' {{
    container = 'ghcr.io/openproblems-bio/task_predict_modality/metrics/correlation:build_main'
    containerOptions = '--env RETICULATE_PYTHON={python_executable} --bind {conda_root}:{conda_root}'
  }}
}}
"""

runtime_config.write_text(config_text)

print("Runtime config:", runtime_config.relative_to(repo_root))
print()
print(config_text)


Runtime config: outputs/tchic/runtime/mean_per_gene.config

process {
  withName: '.*mean_per_gene_process' {
    container = 'openproblems/base_python:1'
  }

  withName: '.*mse_process' {
    container = 'ghcr.io/openproblems-bio/task_predict_modality/metrics/mse:build_main'
  }

  withName: '.*correlation_process' {
    container = 'ghcr.io/openproblems-bio/task_predict_modality/metrics/correlation:build_main'
    containerOptions = '--env RETICULATE_PYTHON=/home/michiel/miniforge3/envs/thesis/bin/python3.10 --bind /home/michiel/miniforge3:/home/michiel/miniforge3'
  }
}



## 4. Construct the exact benchmark command

The benchmark uses the four canonical H5AD files generated by notebook 02.

`NXF_SYNTAX_PARSER=v1` is required because the generated Viash/Nextflow
configuration was produced for the older Nextflow syntax parser.

The exact command is also written to `run_command.sh` for provenance.

In [4]:
command = [
    "nextflow",
    "run",
    str(workflow_dir),
    "-profile",
    "singularity",
    "-c",
    str(runtime_config),
    "-work-dir",
    str(work_dir),
    "-with-trace",
    str(trace_path),

    "--input_train_mod1",
    str(prepared_dir / "train_mod1.h5ad"),

    "--input_train_mod2",
    str(prepared_dir / "train_mod2.h5ad"),

    "--input_test_mod1",
    str(prepared_dir / "test_mod1.h5ad"),

    "--input_test_mod2",
    str(prepared_dir / "test_mod2.h5ad"),

    "--method_ids",
    "mean_per_gene",

    "--output_scores",
    "score_uns.yaml",

    "--output_method_configs",
    "method_configs.yaml",

    "--output_metric_configs",
    "metric_configs.yaml",

    "--output_dataset_info",
    "dataset_uns.yaml",

    "--output_task_info",
    "task_info.yaml",

    "--publish_dir",
    str(publish_dir),
]

shell_command = (
    "NXF_SYNTAX_PARSER=v1 "
    + shlex.join(command)
)

command_path.write_text(
    "#!/usr/bin/env bash\n"
    "set -euo pipefail\n"
    "export NXF_SYNTAX_PARSER=v1\n"
    + shlex.join(command)
    + "\n"
)

command_path.chmod(0o755)

print(shell_command)
print()
print(
    "Saved command:",
    command_path.relative_to(repo_root),
)


NXF_SYNTAX_PARSER=v1 nextflow run /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/target_benchmark_complete/nextflow/workflows/run_benchmark -profile singularity -c /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/runtime/mean_per_gene.config -work-dir /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/runs/h3k4me3_genelevel_dev/mean_per_gene/work -with-trace /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/runs/h3k4me3_genelevel_dev/mean_per_gene/trace.txt --input_train_mod1 /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/prepared/h3k4me3_genelevel_dev/train_mod1.h5ad --input_train_mod2 /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outputs/tchic/prepared/h3k4me3_genelevel_dev/train_mod2.h5ad --input_test_mod1 /home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/outp

## 5. Run the benchmark

This executes the unmodified generated OpenProblems benchmark workflow.

A fresh canonical output location is used rather than the earlier temporary
baseline directories.

In [5]:
env = os.environ.copy()
env["NXF_SYNTAX_PARSER"] = "v1"

result = subprocess.run(
    command,
    cwd=repo_root,
    env=env,
)

if result.returncode != 0:
    raise RuntimeError(
        f"Nextflow benchmark failed with exit code {result.returncode}"
    )

print("BENCHMARK COMPLETED SUCCESSFULLY")


Nextflow 26.04.6 is available - Please consider updating your version to it



 N E X T F L O W   ~  version 26.04.4

Launching `/home/michiel/thesis/benchmark/task_predict_modality_upstream_validation/target_benchmark_complete/nextflow/workflows/run_benchmark/main.nf` [chaotic_sanger] revision: efaca4737c

[-        ] run…ssWf:metadata_mod1_process -

[-        ] run…ssWf:metadata_mod1_process -
[-        ] run…ssWf:metadata_mod2_process -

[-        ] run…ssWf:metadata_mod1_process -
[-        ] run…ssWf:metadata_mod2_process -
[-        ] run…ssWf:mean_per_gene_process -
[-        ] run…sWf:random_predict_process -

[-        ] run…ssWf:metadata_mod1_process -
[-        ] run…ssWf:metadata_mod2_process -
[-        ] run…ssWf:mean_per_gene_process -
[-        ] run…sWf:random_predict_process -
[-        ] run…os:processWf:zeros_process -

[-        ] run…ssWf:metadata_mod1_process -
[-        ] run…ssWf:metadata_mod2_process -
[-        ] run…ssWf:mean_per_gene_process -
[-        ] run…sWf:random_predict_process -
[-        ] run…os:processWf:zeros_process -


## 6. Inspect the benchmark outputs

In [6]:
print("Published files:")

for path in sorted(publish_dir.iterdir()):
    print(
        path.name,
        f"{path.stat().st_size / 1024**2:.2f} MB",
    )


Published files:
dataset_uns.yaml 0.00 MB
method_configs.yaml 0.62 MB
metric_configs.yaml 0.05 MB
output.run_benchmark.state.yaml 0.00 MB
score_uns.yaml 0.00 MB
task_info.yaml 0.00 MB


## 7. Development scores

These values measure performance on the held-out **validation** plates.

They must not be reported as final test performance because the final 15%
partition has not been used in this run.


In [7]:
score_path = publish_dir / "score_uns.yaml"

assert score_path.exists()

with open(score_path) as f:
    scores = yaml.safe_load(f)

print(
    yaml.safe_dump(
        scores,
        sort_keys=False,
    )
)


- dataset_id: tchic_mgast_h3k4me3_genelevel_dev
  date_created: 27-09-2026
  file_size: 22264
  method_id: mean_per_gene
  metric_ids:
  - rmse
  - mae
  metric_values:
  - 0.33553544021715975
  - 0.14435372425485854
- dataset_id: tchic_mgast_h3k4me3_genelevel_dev
  date_created: 27-09-2026
  file_size: 1255833
  method_id: mean_per_gene
  metric_ids:
  - mean_pearson_per_cell
  - mean_spearman_per_cell
  - mean_pearson_per_gene
  - mean_spearman_per_gene
  - overall_pearson
  - overall_spearman
  metric_values:
  - 0.6587230807239869
  - 0.46150140608683904
  - 0.0
  - 0.0
  - 0.6572280062585758
  - 0.46845281271361683



## 8. Benchmark status

At this point:

- training uses only the 70% training plates;
- prediction uses H3K4me3 from the 15% validation plates;
- scoring uses RNA from those same validation plates;
- the final 15% test plates remain untouched;
- the source-generated benchmark workflow has not been modified;
- runtime compatibility settings are recorded explicitly above.

The resulting `mean_per_gene` scores form the canonical development control
baseline for the `log_cp10k` T-ChIC representation.
